In [1]:
import networkx as nx
import pandas as pd
import plotly.graph_objects as go
import random

In [2]:
DATA_PATH = "../data/processed/final_gene_features.csv"
EDGE_PATH = "../data/processed/final_edge_list.csv"

df = pd.read_csv(DATA_PATH)
edges = pd.read_csv(EDGE_PATH)

G = nx.from_pandas_edgelist(
    edges,
    source="gene1",
    target="gene2"
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 16185
Edges: 236000


sample_nodes = random.sample(list(G.nodes()), 600)

G_vis = G.subgraph(sample_nodes)

In [3]:
sample_nodes = random.sample(list(G.nodes()), 600)

G_vis = G.subgraph(sample_nodes)

Compute Node Layout

Use spring layout for clustering.

In [4]:
pos = nx.spring_layout(
    G_vis,
    seed=42
)

Extract Node Positions

In [5]:
x_nodes = []
y_nodes = []

for node in G_vis.nodes():
    x_nodes.append(pos[node][0])
    y_nodes.append(pos[node][1])

Extract Edge Coordinates

In [6]:
x_edges = []
y_edges = []

for edge in G_vis.edges():

    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]

    x_edges += [x0, x1, None]
    y_edges += [y0, y1, None]

Node Degree (Hub Importance)

In [7]:
degree_dict = dict(G.degree())

node_size = [
    degree_dict.get(node,1) * 0.6
    for node in G_vis.nodes()
]

Node Colors (Pathogenic Label)

In [8]:
label_map = dict(
    zip(df["GeneSymbol"], df["label"])
)

node_colors = [
    label_map.get(node,0)
    for node in G_vis.nodes()
]

Create Edge Trace

In [9]:
edge_trace = go.Scatter(
    x=x_edges,
    y=y_edges,
    mode="lines",
    line=dict(width=0.5, color="#888"),
    hoverinfo="none"
)

Create Node Trace

In [10]:
node_trace = go.Scatter(
    x=x_nodes,
    y=y_nodes,
    mode="markers",
    marker=dict(
        size=node_size,
        color=node_colors,
        colorscale="Viridis",
        opacity=0.9
    ),
    text=list(G_vis.nodes()),
    hoverinfo="text"
)

Create Interactive Graph

In [13]:
fig = go.Figure(
    data=[edge_trace, node_trace]
)

fig.update_layout(
    title="Gene Interaction Network (2D)",
    showlegend=False,
    hovermode="closest",
    margin=dict(l=0,r=0,b=0,t=40)
)

fig.show()